# Sila i hvatište na pravokutnu plohu pod vodom

**Poglavlje U05: Hidrostatske sile na ravne plohe**

Ovaj interaktivni prikaz nadopunjuje izvod hidrostatičke sile na ravnu uronjenu plohu. Mijenjanjem dubine gornjeg ruba, visine plohe i kuta nagiba prati se ukupna sila i položaj hvatišta.

## Cilj

Na ravnoj uronjenoj plohi hidrostatička sila proizlazi iz integralnog dijela tlaka po plohi, a njezino hvatište spušteno je ispod težišta plohe. Prikaz omogućuje:

1. mijenjanje dubine gornjeg ruba plohe $h_t$;
2. mijenjanje visine plohe $b$;
3. mijenjanje kuta plohe prema vertikali $\alpha$;
4. praćenje sile, dubine težišta i položaja hvatišta.

## Pretpostavke modela

- jednolika gustoća vode, $\rho = 998$ kg/m³;
- pravokutna ploha jedinične širine ($L = 1$ m);
- statičko stanje, bez strujanja;
- na slobodnoj površini atmosferski tlak (manometarski tlak nula).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import interact, FloatSlider, Layout

plt.rcParams['figure.dpi'] = 110
plt.rcParams['font.size'] = 10

## Računski model

Dubina težišta uronjene plohe iznosi:

$$h_c = h_t + \frac{b}{2}\cos\alpha.$$

Ukupna sila na plohu:

$$F = \rho g A h_c, \qquad A = b \cdot L.$$

Položaj hvatišta sile, mjeren po plohi od slobodne površine:

$$y_{CP} = \frac{h_c}{\cos\alpha} + \frac{I_{xc}\cos\alpha}{A\,h_c},$$

gdje je $I_{xc} = L b^3 / 12$ aksijalni moment tromosti pravokutnika oko vlastite težišne osi.

In [ ]:
RHO = 998.0
G = 9.81
L = 1.0  # jedinična širina plohe (m)

def ploha(h_t, b, alpha_deg):
    alpha = np.radians(alpha_deg)
    h_c = h_t + (b / 2) * np.cos(alpha)
    A = b * L
    F = RHO * G * A * h_c
    I_xc = L * b**3 / 12
    if h_c > 0 and np.cos(alpha) != 0:
        y_cp = h_c / np.cos(alpha) + (I_xc * np.cos(alpha)) / (A * h_c)
    else:
        y_cp = float('nan')
    return {'F': F, 'h_c': h_c, 'y_cp': y_cp, 'A': A}

## Interaktivni prikaz

Klizačima u nastavku biraju se dubina gornjeg ruba, visina plohe i kut nagiba. Prikaz pokazuje bočni presjek plohe pod vodom uz dijagram tlaka i položaj hvatišta sile.

In [ ]:
def ploha_prikaz(h_t, b, alpha_deg):
    r = ploha(h_t, b, alpha_deg)
    alpha = np.radians(alpha_deg)

    fig, ax = plt.subplots(figsize=(9, 6))

    # Slobodna površina
    ax.axhline(0, color='#1565c0', lw=2, label='slobodna površina')
    ax.fill_between([-1, 3], 0, -max(2*(h_t+b), 2),
                     fc='#aed6f1', alpha=0.3)

    # Ploha — bočni presjek
    x_top = 1.0
    y_top = -h_t
    x_bot = x_top + b * np.sin(alpha)
    y_bot = -h_t - b * np.cos(alpha)
    ax.plot([x_top, x_bot], [y_top, y_bot],
             color='#c62828', lw=4, label='ploha')

    # Dijagram tlaka uz plohu
    n = 30
    s = np.linspace(0, b, n)  # koordinata uz plohu
    h_uz = h_t + s * np.cos(alpha)
    p_uz = RHO * G * h_uz / 1000  # u kPa
    x_p = x_top + s * np.sin(alpha)
    y_p = y_top - s * np.cos(alpha)
    # Strelice tlaka (normalne na plohu)
    skala = 0.005  # m po kPa
    for i in range(0, n, 3):
        n_x = np.cos(alpha)  # normala plohe (vanjska)
        n_y = np.sin(alpha)
        L_str = p_uz[i] * skala
        ax.annotate('', xy=(x_p[i], y_p[i]),
                     xytext=(x_p[i] + L_str * n_x,
                              y_p[i] + L_str * n_y),
                     arrowprops=dict(arrowstyle='->',
                                       color='#c62828', alpha=0.6))

    # Hvatište sile
    if not np.isnan(r['y_cp']):
        s_cp = r['y_cp'] - h_t / np.cos(alpha)
        if 0 <= s_cp <= b:
            x_cp = x_top + s_cp * np.sin(alpha)
            y_cp = y_top - s_cp * np.cos(alpha)
            ax.scatter([x_cp], [y_cp], color='#2e7d32',
                        s=120, zorder=5, marker='X',
                        label='hvatište $y_{CP}$')

    ax.set_xlim(-0.3, 3)
    ax.set_ylim(-(h_t + b + 0.5), 0.5)
    ax.set_aspect('equal')
    ax.set_xlabel('horizontalna koordinata (m)')
    ax.set_ylabel('dubina (m)')
    ax.set_title(
        f'$h_t$ = {h_t:.2f} m,  $b$ = {b:.2f} m,  '
        f'$\\alpha$ = {alpha_deg:.0f}°\n'
        f'$F$ = {r["F"]/1000:.2f} kN,  '
        f'$h_c$ = {r["h_c"]:.3f} m,  '
        f'$y_{{CP}}$ = {r["y_cp"]:.3f} m'
    )
    ax.legend(loc='lower right', fontsize=9)
    ax.grid(ls=':', alpha=0.5)

    plt.tight_layout()
    plt.show()


interact(
    ploha_prikaz,
    h_t=FloatSlider(min=0, max=4, step=0.1, value=0.5,
                     description='$h_t$ (m)',
                     layout=Layout(width='420px')),
    b=FloatSlider(min=0.2, max=3, step=0.1, value=1.5,
                   description='$b$ (m)',
                   layout=Layout(width='420px')),
    alpha_deg=FloatSlider(min=0, max=80, step=5, value=30,
                           description='$\\alpha$ (°)',
                           layout=Layout(width='420px'))
);

## Pitanja za istraživanje

1. **Položaj hvatišta.** Hvatište sile uvijek leži ispod težišta plohe. Kako se ta razlika mijenja kada $h_t$ raste, a $b$ i $\alpha$ ostaju nepromijenjeni? Zašto se hvatište približava težištu pri velikim dubinama?

2. **Vertikalna i nagnuta ploha.** Pri istoj površini i istoj dubini težišta, daje li okomita ploha veću ili manju silu od nagnute? A drukčiji položaj hvatišta?

3. **Skala s visinom plohe.** Provjeri da pri udvostručenoj visini $b$ (uz konstantne $h_t$ i $\alpha$) sila raste više od dva puta. Koji je razlog?

4. **Inženjerska procjena.** Za vrata brane s $h_t = 0$ i $b = 5$ m, okomito uronjena, kolika je sila po metru širine? Gdje leži hvatište?

## Veza s teorijom poglavlja

Ovaj prikaz materijalizira dva temeljna rezultata iz poglavlja U05: ukupna hidrostatička sila ovisi o površini i dubini težišta plohe, dok je hvatište te sile uvijek spušteno ispod težišta zbog linearnog rasta tlaka s dubinom. U poglavlju U06 isti se aparat proširuje na zakrivljene plohe gdje se sila razlaže na horizontalnu i vertikalnu komponentu.